# 02 — Limpieza ESS para la muestra analítica EPBI

> **Pipeline:** 01 Auditoría → **02 Limpieza ESS** → 03 Eurostat → 04 Construcción EPBI → 05 Integración micro–macro → 06 Econometría → 07 Datos para dashboard

Este notebook aplica las decisiones obtenidas en la auditoría del notebook 01 y convierte los microdatos ESS brutos en **una única muestra limpia y elegible para el análisis del EPBI**.

La función de esta etapa es cerrar la preparación de los microdatos: seleccionar el periodo analítico, normalizar valores no sustantivos, validar rangos, construir el peso de análisis y preservar la información necesaria para las etapas posteriores. **El EPBI todavía no se calcula aquí**; su construcción se realizará en el notebook 04.

## Decisiones metodológicas aplicadas

1. El análisis del EPBI comienza en la **ronda 4 (2008)**.
2. Las rondas 1–3 se excluyen de la muestra analítica porque `hinctnta` no está disponible.
3. Desde la ronda 4, se excluye cualquier combinación país-ronda sin ninguna observación válida para construir el EPBI.
4. Los códigos especiales de no respuesta se convierten a `NA` variable por variable.
5. `eisced = 0` se conserva como categoría válida: **Menor que educación primaria**.
6. Se mantienen las variables originales ESS necesarias y se añaden identificadores y etiquetas de apoyo.
7. `analysis_weight` se construye a partir de los pesos ESS disponibles; nunca se sustituye por un peso unitario.
8. `psu` y `stratum` se conservan por trazabilidad y para posibles análisis de sensibilidad o extensiones, aunque el modelo econométrico final no utiliza PSU.
9. La única salida de esta fase es `ess_clean_epbi_sample.parquet`.

La salida de este notebook volverá a utilizarse en el **notebook 04**, después de que el notebook 03 prepare en paralelo el contexto macroeconómico de Eurostat.


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

BRUTOS = PROJECT_ROOT / "DATOS" / "BRUTOS"
PROCESADOS = PROJECT_ROOT / "DATOS" / "PROCESADOS"

PROCESADOS.mkdir(parents=True, exist_ok=True)

INPUT = BRUTOS / "ESS11_microdatos.csv"

OUT_PARQUET = PROCESADOS / "ess_clean_epbi_sample.parquet"

print("Entrada:", INPUT)


## 1. Variables necesarias y carga

Partiendo del diagnóstico del notebook 01, se seleccionan las variables ESS que deben acompañar a la muestra a lo largo del proyecto y se carga únicamente esa información. Así se reduce el volumen de trabajo sin perder las variables necesarias para construir el índice, los pesos, los controles y los modelos posteriores.


In [ ]:

VARIABLES = [
    # Identificación
    "name", "essround", "edition", "proddate", "idno", "cntry",

    # Pesos
    "dweight", "pspwght", "pweight", "anweight", "psu", "stratum",

    # Variables económicas / hogar
    "hinctnta", "hincfel", "hhmmb",

    # Sociodemográficas
    "agea", "gndr", "eisced", "eduyrs", "mnactic",

    # Actitudinales
    "lrscale", "gincdif", "ppltrst", "trstprl", "trstplt",
    "stfdem", "stfeco", "stflife",
]

if not INPUT.exists():
    raise FileNotFoundError(f"No se encuentra el fichero de entrada: {INPUT}")

header = pd.read_csv(INPUT, nrows=0, sep=",", encoding="utf-8-sig")
present = [c for c in VARIABLES if c in header.columns]
missing = [c for c in VARIABLES if c not in header.columns]

print("Variables solicitadas ausentes:", missing)

ess = pd.read_csv(
    INPUT,
    sep=",",
    encoding="utf-8-sig",
    usecols=present,
    low_memory=True,
)

print("Dimensiones cargadas:", ess.shape)
display(ess.head())


## 2. Identificadores y periodo analítico

Con los datos cargados, se normalizan los identificadores y se deriva el año de encuesta a partir de la ronda ESS. En este mismo punto se aplica la primera decisión metodológica de la auditoría: la muestra analítica se restringe a las rondas **4 en adelante**.


In [ ]:

required = {"essround", "cntry", "idno", "hinctnta", "hincfel"}
absent = sorted(required - set(ess.columns))

if absent:
    raise KeyError("Faltan variables obligatorias: " + ", ".join(absent))

ess["essround"] = pd.to_numeric(ess["essround"], errors="coerce").astype("Int64")
ess["idno"] = pd.to_numeric(ess["idno"], errors="coerce").astype("Int64")
ess["cntry"] = ess["cntry"].astype("string").str.strip().str.upper()

ROUND_YEAR = {
    1: 2002, 2: 2004, 3: 2006, 4: 2008, 5: 2010, 6: 2012,
    7: 2014, 8: 2016, 9: 2018, 10: 2020, 11: 2023
}

ess["survey_year"] = ess["essround"].map(ROUND_YEAR).astype("Int64")
ess["country_round_id"] = (
    ess["cntry"].astype("string") + "_R" + ess["essround"].astype("string")
)
ess["respondent_id"] = (
    ess["country_round_id"] + "_" + ess["idno"].astype("string")
)

# El análisis EPBI comienza en la ronda 4.
ess = ess.loc[ess["essround"] >= 4].copy()

print("Dimensiones tras excluir rondas 1-3:", ess.shape)
print("Rondas conservadas:", sorted(ess["essround"].dropna().unique().tolist()))


## 3. Control de duplicados

Antes de recodificar variables se comprueba que cada entrevistado queda identificado de forma única. La limpieza solo continúa si `respondent_id` no presenta duplicados, evitando que una misma persona pueda propagarse varias veces por el pipeline.


In [ ]:

dup_mask = ess["respondent_id"].duplicated(keep=False)
n_dup_extra = int(ess["respondent_id"].duplicated().sum())

print("Duplicados adicionales de respondent_id:", n_dup_extra)

if n_dup_extra > 0:
    display(
        ess.loc[dup_mask, ["cntry", "essround", "idno", "respondent_id"]]
        .sort_values(["cntry", "essround", "idno"])
        .head(100)
    )
    raise ValueError(
        "Se han detectado respondent_id duplicados. "
        "El notebook no los elimina automáticamente."
    )


## 4. Conversión de códigos especiales a `NA`

Una vez validada la identificación, se transforman los códigos especiales de la ESS que no representan respuestas sustantivas. Estos valores se convierten a `NA` de forma explícita y variable por variable.

`eisced = 0` **no** se recodifica, porque representa una categoría válida: **Menor que educación primaria**. Esta distinción entre códigos sustantivos y códigos especiales procede directamente de la auditoría realizada en el notebook 01.


In [ ]:

SPECIAL_CODES = {
    "hinctnta": [77, 88, 99],
    "hincfel": [7, 8, 9],
    "hhmmb": [77, 88, 99],
    "agea": [999],
    "gndr": [9],
    "eisced": [55, 77, 88, 99],   # 0 = Menor que educación primaria
    "eduyrs": [77, 88, 99],
    "mnactic": [77, 88, 99],
    "lrscale": [77, 88, 99],
    "gincdif": [7, 8, 9],
    "ppltrst": [77, 88, 99],
    "trstprl": [77, 88, 99],
    "trstplt": [77, 88, 99],
    "stfdem": [77, 88, 99],
    "stfeco": [77, 88, 99],
    "stflife": [77, 88, 99],
}

recoding_log = []

for var, codes in SPECIAL_CODES.items():
    if var not in ess.columns:
        continue

    s = pd.to_numeric(ess[var], errors="coerce")
    n_special = int(s.isin(codes).sum())

    recoding_log.append({
        "variable": var,
        "codigos_a_NA": ", ".join(map(str, codes)),
        "n_reemplazos": n_special,
    })

    ess[var] = s.mask(s.isin(codes), np.nan)

recoding_log = pd.DataFrame(recoding_log)
display(recoding_log)


## 5. Validación de rangos sustantivos

Después de retirar los códigos especiales, se comprueba que los valores restantes se encuentran dentro de los rangos admitidos para cada variable. Cualquier valor fuera de rango se trata como ausente, de modo que las etapas posteriores trabajen únicamente con codificaciones coherentes.


In [ ]:

VALID_RANGES = {
    "hinctnta": (1, 10),
    "hincfel": (1, 4),
    "gndr": (1, 2),
    "eisced": (0, 7),
    "mnactic": (1, 9),
    "lrscale": (0, 10),
    "gincdif": (1, 5),
    "ppltrst": (0, 10),
    "trstprl": (0, 10),
    "trstplt": (0, 10),
    "stfdem": (0, 10),
    "stfeco": (0, 10),
    "stflife": (0, 10),
}

validation_log = []

for var, (lo, hi) in VALID_RANGES.items():
    if var not in ess.columns:
        continue

    s = pd.to_numeric(ess[var], errors="coerce")
    out_of_range = s.notna() & ~s.between(lo, hi)

    validation_log.append({
        "variable": var,
        "rango_valido": f"{lo}-{hi}",
        "n_fuera_rango": int(out_of_range.sum()),
    })

    if out_of_range.any():
        ess.loc[out_of_range, var] = np.nan

validation_log = pd.DataFrame(validation_log)
display(validation_log)


## 6. Construcción del peso de análisis según la guía ESS

Con las variables sustantivas ya depuradas, se construye `analysis_weight`, que será el peso utilizado en los descriptivos ponderados y en la estimación WLS del notebook 06.

La regla es:

1. utilizar `anweight` cuando está disponible y es positivo;
2. si falta, reconstruirlo como:

```python
analysis_weight = pspwght * pweight
```

No se sustituye por peso 1. Si no existen componentes suficientes para obtener un peso válido, `analysis_weight` permanece como `NA` y el caso queda identificado.

`psu` y `stratum` se conservan sin modificarlos para mantener la trazabilidad del diseño muestral. En la especificación econométrica final no se utiliza clustering por PSU; esa decisión se documenta y evalúa en el notebook 06.


In [ ]:
# Convertir pesos a numérico
for var in ["anweight", "pspwght", "pweight", "dweight"]:
    if var in ess.columns:
        ess[var] = pd.to_numeric(ess[var], errors="coerce")

# Validar pesos: solo valores estrictamente positivos
for var in ["anweight", "pspwght", "pweight", "dweight"]:
    if var in ess.columns:
        ess.loc[ess[var] <= 0, var] = np.nan

# Peso de análisis ESS
ess["analysis_weight"] = np.nan
ess["analysis_weight_source"] = pd.Series(pd.NA, index=ess.index, dtype="string")

# 1) Usar anweight si ya está disponible y es válido
mask_an = ess["anweight"].notna() if "anweight" in ess.columns else pd.Series(False, index=ess.index)
ess.loc[mask_an, "analysis_weight"] = ess.loc[mask_an, "anweight"]
ess.loc[mask_an, "analysis_weight_source"] = "anweight"

# 2) Si falta anweight, reconstruirlo como pspwght * pweight
if {"pspwght", "pweight"}.issubset(ess.columns):
    mask_rebuild = (
        ess["analysis_weight"].isna()
        & ess["pspwght"].notna()
        & ess["pweight"].notna()
        & (ess["pspwght"] > 0)
        & (ess["pweight"] > 0)
    )

    ess.loc[mask_rebuild, "analysis_weight"] = (
        ess.loc[mask_rebuild, "pspwght"]
        * ess.loc[mask_rebuild, "pweight"]
    )
    ess.loc[mask_rebuild, "analysis_weight_source"] = "pspwght * pweight"
else:
    mask_rebuild = pd.Series(False, index=ess.index)

weight_summary = (
    ess["analysis_weight_source"]
    .fillna("sin_peso_valido")
    .value_counts(dropna=False)
    .rename_axis("fuente_peso")
    .reset_index(name="n")
)

weight_summary["pct"] = (100 * weight_summary["n"] / len(ess)).round(2)

print("Distribución de la fuente de analysis_weight:")
display(weight_summary)

n_missing_weight = int(ess["analysis_weight"].isna().sum())
print("Casos sin analysis_weight válido:", n_missing_weight)

if n_missing_weight > 0:
    print(
        "ATENCIÓN: estos casos no reciben peso 1. "
        "Quedan con analysis_weight = NA para revisión posterior."
    )

## 7. Etiquetas de presentación

Con la parte analítica de la limpieza cerrada, se añaden etiquetas legibles para facilitar la inspección y la posterior preparación del dashboard. Las variables originales se conservan, por lo que estas etiquetas complementan la información sin sustituir las codificaciones ESS.


In [ ]:

if "gndr" in ess.columns:
    ess["sexo"] = ess["gndr"].map({
        1: "Hombre",
        2: "Mujer",
    }).astype("string")

if "hincfel" in ess.columns:
    ess["percepcion_ingresos"] = ess["hincfel"].map({
        1: "Vive cómodamente con los ingresos actuales",
        2: "Se las arregla con los ingresos actuales",
        3: "Le resulta difícil vivir con los ingresos actuales",
        4: "Le resulta muy difícil vivir con los ingresos actuales",
    }).astype("string")

if "hinctnta" in ess.columns:
    ess["decil_renta"] = ess["hinctnta"].map(
        {i: f"Decil {i}" for i in range(1, 11)}
    ).astype("string")

if "eisced" in ess.columns:
    EISCED_LABELS = {
        0: "Menor que educación primaria",
        1: "Educación primaria",
        2: "Educación baja o primera etapa de secundaria",
        3: "Educación secundaria superior",
        4: "Educación post-secundaria no terciaria",
        5: "Educación terciaria de ciclo corto",
        6: "Grado universitario o equivalente",
        7: "Maestría, Doctorado o Educación Superior",
    }
    ess["nivel_educativo"] = ess["eisced"].map(EISCED_LABELS).astype("string")

if "mnactic" in ess.columns:
    ess["actividad_principal"] = ess["mnactic"].map({
        1: "Trabajo remunerado",
        2: "Educación",
        3: "Desempleo, buscando activamente",
        4: "Desempleo, sin búsqueda activa",
        5: "Enfermedad o discapacidad permanente",
        6: "Jubilación",
        7: "Servicio comunitario o militar",
        8: "Tareas domésticas o cuidado",
        9: "Otra actividad",
    }).astype("string")

display(
    ess[
        [c for c in [
            "gndr", "sexo",
            "hincfel", "percepcion_ingresos",
            "hinctnta", "decil_renta",
            "eisced", "nivel_educativo",
            "mnactic", "actividad_principal"
        ] if c in ess.columns]
    ].head(10)
)


## 8. Detección y exclusión de país-ronda sin datos para EPBI

A continuación se aplica la segunda decisión principal derivada del notebook 01: una combinación país-ronda solo permanece en la muestra si contiene al menos un caso con ambos inputs potencialmente válidos:

- `hinctnta` entre 1 y 10;
- `hincfel` entre 1 y 4.

Las combinaciones con **0 casos potencialmente válidos** se eliminan porque no pueden aportar ninguna observación al cálculo del EPBI.


In [ ]:

ess["epbi_input_valid"] = (
    pd.to_numeric(ess["hinctnta"], errors="coerce").between(1, 10)
    & pd.to_numeric(ess["hincfel"], errors="coerce").between(1, 4)
)

country_round_availability = (
    ess.groupby(["cntry", "essround", "survey_year"], dropna=False)
    .agg(
        n_total=("respondent_id", "size"),
        epbi_inputs_validos=("epbi_input_valid", "sum"),
    )
    .reset_index()
)

country_round_availability["pct_epbi_inputs_validos"] = (
    100
    * country_round_availability["epbi_inputs_validos"]
    / country_round_availability["n_total"]
).round(2)

excluded_country_rounds = country_round_availability[
    country_round_availability["epbi_inputs_validos"].eq(0)
].copy()

excluded_country_rounds["motivo_exclusion"] = (
    "País-ronda excluido por ausencia total de datos válidos "
    "para construir el EPBI"
)

print("País-ronda excluidos:", len(excluded_country_rounds))
display(
    excluded_country_rounds.sort_values(["survey_year", "cntry"])
)

excluded_ids = set(
    excluded_country_rounds["cntry"].astype("string")
    + "_R"
    + excluded_country_rounds["essround"].astype("string")
)

ess = ess.loc[
    ~ess["country_round_id"].isin(excluded_ids)
].copy()

print("Dimensiones tras excluir país-ronda sin datos EPBI:", ess.shape)


## 9. Controles finales

Antes de guardar la muestra se verifican conjuntamente las decisiones anteriores: periodo analítico, unicidad de identificadores, rangos válidos, construcción del peso y conservación de las variables de diseño.

El dataset mantiene **todos los individuos pertenecientes a las combinaciones país-ronda elegibles**, incluso cuando a una persona concreta le falte `hinctnta` o `hincfel`. La validez individual del EPBI se determinará en el notebook 04.


In [ ]:

checks = pd.DataFrame([
    {
        "control": "Ronda mínima = 4",
        "resultado": "OK" if ess["essround"].min() >= 4 else "REVISAR"
    },
    {
        "control": "Sin duplicados respondent_id",
        "resultado": "OK" if ess["respondent_id"].duplicated().sum() == 0 else "REVISAR"
    },
    {
        "control": "hinctnta en rango 1-10 o NA",
        "resultado": "OK" if ess["hinctnta"].dropna().between(1, 10).all() else "REVISAR"
    },
    {
        "control": "hincfel en rango 1-4 o NA",
        "resultado": "OK" if ess["hincfel"].dropna().between(1, 4).all() else "REVISAR"
    },
    {
        "control": "eisced=0 conservado",
        "resultado": "OK" if (ess["eisced"] == 0).sum() > 0 else "REVISAR"
    },
    {
        "control": "analysis_weight válido o NA",
        "resultado": "OK" if ess["analysis_weight"].dropna().gt(0).all() else "REVISAR"
    },
    {
        "control": "psu conservado",
        "resultado": "OK" if "psu" in ess.columns else "REVISAR"
    },
    {
        "control": "stratum conservado",
        "resultado": "OK" if "stratum" in ess.columns else "REVISAR"
    },
])

display(checks)

print("Filas finales:", len(ess))
print("Países:", ess["cntry"].nunique())
print("Rondas:", sorted(ess["essround"].dropna().unique().tolist()))
print("País-ronda:", ess["country_round_id"].nunique())

print("\nCasos con inputs EPBI completos:",
      int(ess["epbi_input_valid"].sum()))
print("Casos sin inputs EPBI completos:",
      int((~ess["epbi_input_valid"]).sum()))


print("Casos sin analysis_weight:", int(ess["analysis_weight"].isna().sum()))


## 10. Exportación del dataset limpio

Superados los controles, la muestra limpia se guarda en formato Parquet. Se genera una única salida analítica para evitar versiones intermedias redundantes y asegurar que el notebook 04 parta siempre de la misma base.


In [ ]:
# Se genera una única salida en formato Parquet.
# Parquet es compacto, rápido y conserva correctamente los tipos de datos.
ess.to_parquet(OUT_PARQUET, index=False)

print("Dataset limpio ESS para EPBI:")
print(" -", OUT_PARQUET)


## 11. Resultado de la fase 02 y continuidad

La salida de esta etapa es:

`DATOS/PROCESADOS/ess_clean_epbi_sample.parquet`

El fichero contiene:

- únicamente rondas desde 2008;
- únicamente combinaciones país-ronda con disponibilidad potencial para construir el EPBI;
- códigos especiales convertidos a `NA`;
- `eisced = 0` conservado como categoría válida;
- identificadores consistentes;
- variables técnicas ESS y etiquetas de presentación;
- `analysis_weight`, obtenido de `anweight` o, cuando es necesario, de `pspwght * pweight`;
- `psu` y `stratum`, conservados por trazabilidad.

Con esta fase queda cerrada la limpieza de la ESS. El **notebook 03** prepara por separado los indicadores Eurostat y el **notebook 04** retoma este Parquet para construir el EPBI individual. Ambas ramas se unirán posteriormente en el notebook 05.
